# Cross-Asset Skew Strategy (yfinance Replication)

This notebook replicates the cross-asset skew strategy workflow using free Yahoo Finance data via `yfinance`.

Pipeline:
1. Load ETF universe data
2. Compute rolling skew signal
3. Build monthly cross-sectional long/short weights within asset class
4. Backtest daily portfolio returns
5. Volatility-normalize and aggregate a global factor
6. Run alpha/beta regressions and equity factor regression

In [ ]:
import datetime as dt
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.float_format = "{:.6f}".format

In [ ]:
# Strategy parameters
LOOKBACK = 256
HISTORY_DAYS = 3650  # ~10 years
VOL_TARGET = 0.10

# Universe (inspired by experiments/cam_skewness_v1.ipynb)
universe = [
    ("Equity", ["SPY", "EWU", "EWJ", "INDA", "EWG", "EWL", "EWP", "EWQ",
                "VTI", "FXI", "EWZ", "EWY", "EWA", "EWC", "EWG",
                "EWH", "EWI", "EWN", "EWD", "EWT", "EZA", "EWW", "ENOR", "EDEN", "TUR"]),
    ("FI", ["AGG", "TLT", "LQD", "JNK", "MUB", "MBB", "IAGG", "IGOV", "EMB",
            "BND", "BNDX", "VCIT", "VCSH", "BSV", "SRLN"]),
    ("Commodities", ["GLD", "SLV", "GSG", "USO", "PPLT", "UNG", "DBA"]),
    ("Other", ["IYR", "REET", "USRT", "ICF", "VNQ"]),
    ("Ccy", ["UUP", "FXY", "FXE", "FXF", "FXB", "FXA", "FXC"]),
]

In [ ]:
def clean_yf(raw: pd.DataFrame, ticker: str) -> pd.DataFrame:
    base_cols = ["Date", "Ticker", "close", "open", "NextOpen", "LogReturn"]

    if raw is None or raw.empty:
        return pd.DataFrame(columns=base_cols)

    df = raw.copy()

    # yfinance may return MultiIndex columns (e.g., ("Adj Close", "SPY")) depending on version.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

    df = df.reset_index()
    if "Date" not in df.columns and "Datetime" in df.columns:
        df = df.rename(columns={"Datetime": "Date"})

    if "Date" not in df.columns:
        return pd.DataFrame(columns=base_cols)

    if "Open" not in df.columns:
        return pd.DataFrame(columns=base_cols)

    close_col = "Adj Close" if "Adj Close" in df.columns else ("Close" if "Close" in df.columns else None)
    if close_col is None:
        return pd.DataFrame(columns=base_cols)

    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    df["Ticker"] = ticker
    df["close"] = pd.to_numeric(df[close_col], errors="coerce")
    df["open"] = pd.to_numeric(df["Open"], errors="coerce")

    df = df.sort_values("Date").drop_duplicates(subset=["Date"], keep="last")
    df["NextOpen"] = df["open"].shift(-1)
    df["LogReturn"] = np.log(df["close"] / df["close"].shift(1))

    out = df[base_cols].dropna(subset=["close", "open"])
    return out


def load_ticker_yf(ticker: str, start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    raw = yf.download(
        ticker,
        start=start_date.date(),
        end=end_date.date(),
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    return clean_yf(raw, ticker)


def load_universe_yf(universe_spec):
    end_date = pd.Timestamp(dt.datetime.now().date()) + pd.Timedelta(days=1)
    start_date = end_date - pd.Timedelta(days=HISTORY_DAYS)

    frames = []
    failures = []

    for asset_class, tickers in universe_spec:
        for ticker in sorted(set(tickers)):  # de-duplicate repeated tickers like EWG
            try:
                df = load_ticker_yf(ticker, start_date, end_date)
                if df.empty:
                    failures.append((asset_class, ticker, "empty or missing required columns"))
                    continue
                df["AssetClass"] = asset_class
                frames.append(df)
            except Exception as exc:
                failures.append((asset_class, ticker, str(exc)))

    all_cols = ["Date", "Ticker", "close", "open", "NextOpen", "LogReturn", "AssetClass"]

    if frames:
        all_data = pd.concat(frames, ignore_index=True)
        all_data = all_data.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    else:
        all_data = pd.DataFrame(columns=all_cols)

    if failures:
        print(f"Failed tickers: {len(failures)}")
        print(pd.DataFrame(failures, columns=["AssetClass", "Ticker", "Error"]).head(50))

    if all_data.empty:
        print("No data loaded. Check internet connection, yfinance version, or ticker availability.")

    return all_data



In [ ]:
all_data = load_universe_yf(universe)
print(all_data.shape)
all_data.head()

In [ ]:
def add_skew_features(df: pd.DataFrame, lookback: int = LOOKBACK) -> pd.DataFrame:
    df = df.copy().sort_values(["Ticker", "Date"])

    def per_ticker(g):
        g = g.sort_values("Date").copy()
        g["Avg"] = g["LogReturn"].rolling(window=lookback, min_periods=lookback).mean()
        g["Dev"] = g["LogReturn"].rolling(window=lookback, min_periods=lookback).std()
        g["SkewDay"] = ((g["LogReturn"] - g["Avg"]) / g["Dev"]) ** 3
        g["Skew"] = g["SkewDay"].rolling(window=lookback, min_periods=lookback).mean()
        return g

    out = df.groupby("Ticker", group_keys=False).apply(per_ticker)
    out = out.dropna(subset=["Skew"]).reset_index(drop=True)
    return out


all_data = add_skew_features(all_data)
print(all_data.shape)
all_data.head()

In [ ]:
# Quick diagnostics: cumulative returns and skew for sample ETFs
sample_tickers = ["SPY", "GLD", "AGG"]
sample = all_data[all_data["Ticker"].isin(sample_tickers)].copy()

plt.figure(figsize=(12, 4))
for t in sample_tickers:
    s = sample[sample["Ticker"] == t]
    if not s.empty:
        plt.plot(s["Date"], s["LogReturn"].cumsum(), label=t)
plt.title("Cumulative Log Returns")
plt.xlabel("Date")
plt.ylabel("Cumulative Log Return")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
for t in sample_tickers:
    s = sample[sample["Ticker"] == t]
    if not s.empty:
        plt.plot(s["Date"], s["Skew"], label=t)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Rolling Skew")
plt.xlabel("Date")
plt.ylabel("Skew")
plt.legend()
plt.show()

## Backtest Construction

- Rebalance monthly using end-of-month skew signal.
- Within each asset class, rank by skew (most negative skew gets positive weight).
- Normalize weights so longs sum to `+1` and shorts sum to `-1` per asset class.
- Apply weights from the next trading day onward until next rebalance.

In [ ]:
bt = all_data.copy().sort_values(["Ticker", "Date"]).reset_index(drop=True)

bt["Month"] = bt["Date"].dt.to_period("M").dt.to_timestamp()
bt["NextDay"] = bt.groupby("Ticker")["Date"].shift(-1)

# Place final missing next-day beyond sample end.
bt["NextDay"] = bt["NextDay"].fillna(bt["Date"] + pd.offsets.BDay(1))

monthly_vals = (
    bt.groupby(["Month", "AssetClass", "Ticker"], as_index=False)
      .agg(Date=("Date", "last"), NextDate=("NextDay", "last"), EOMSkew=("Skew", "last"))
)

monthly_vals["SkewWeightRaw"] = monthly_vals.groupby(["Date", "AssetClass"])["EOMSkew"].transform(
    lambda s: s.rank(ascending=False, method="average") - ((len(s) + 1) / 2)
)

def normalize_raw_weights(s: pd.Series) -> pd.Series:
    denom = s.abs().sum() / 2
    if denom == 0 or pd.isna(denom):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return s / denom

monthly_vals["SkewWeight"] = monthly_vals.groupby(["Date", "AssetClass"])["SkewWeightRaw"].transform(normalize_raw_weights)

monthly_vals.head()

In [ ]:
# Join monthly weights on their activation date (next trading day), then forward-fill within ticker
weightings = monthly_vals[["NextDate", "Ticker", "SkewWeight"]].rename(columns={"NextDate": "Date"})

all_data_weights = bt.merge(weightings, on=["Date", "Ticker"], how="left")
all_data_weights = all_data_weights.sort_values(["Ticker", "Date"]).reset_index(drop=True)
all_data_weights["SkewWeightFF"] = all_data_weights.groupby("Ticker")["SkewWeight"].ffill()

all_data_weights[["Date", "Ticker", "AssetClass", "LogReturn", "SkewWeight", "SkewWeightFF"]].head()

In [ ]:
# Daily per-asset-class returns
all_data_weights["WeightedReturn"] = all_data_weights["SkewWeightFF"] * all_data_weights["LogReturn"]

asset_portfolios = (
    all_data_weights.groupby(["Date", "AssetClass"], as_index=False)
    .agg(
        PortfolioReturn=("WeightedReturn", "sum"),
        MktReturn=("LogReturn", "mean"),
    )
    .dropna()
)

asset_portfolios.head()

In [ ]:
# Plot cumulative returns by asset class
plt.figure(figsize=(12, 6))
for ac in sorted(asset_portfolios["AssetClass"].unique()):
    ac_data = asset_portfolios[asset_portfolios["AssetClass"] == ac]
    plt.plot(ac_data["Date"], ac_data["PortfolioReturn"].cumsum(), label=ac)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Skew Portfolios by Asset Class")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.show()

In [ ]:
# Volatility normalization and global factor
asset_portfolios = asset_portfolios.sort_values(["AssetClass", "Date"]).reset_index(drop=True)
asset_portfolios["Vol"] = asset_portfolios.groupby("AssetClass")["PortfolioReturn"].transform(
    lambda s: s.rolling(window=LOOKBACK, min_periods=100).std()
)

asset_portfolios["NormReturn"] = VOL_TARGET * asset_portfolios["PortfolioReturn"] / asset_portfolios["Vol"]
asset_portfolios["NormMarketReturn"] = VOL_TARGET * asset_portfolios["MktReturn"] / asset_portfolios["Vol"]

gcf = (
    asset_portfolios.dropna(subset=["NormReturn", "NormMarketReturn"])
    .groupby("Date", as_index=False)
    .agg(Return=("NormReturn", "mean"), MktReturn=("NormMarketReturn", "mean"))
)

plt.figure(figsize=(12, 6))
plt.plot(gcf["Date"], gcf["Return"].cumsum(), label="Global Skew Factor")
plt.plot(gcf["Date"], gcf["MktReturn"].cumsum(), label="Global Market Return")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Global Portfolio")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.show()

In [ ]:
# Alpha/Beta regression by asset class
rows = []
for ac in sorted(asset_portfolios["AssetClass"].unique()):
    ac_data = asset_portfolios[asset_portfolios["AssetClass"] == ac].dropna(subset=["PortfolioReturn", "MktReturn"])
    X = sm.add_constant(ac_data["MktReturn"])
    y = ac_data["PortfolioReturn"]
    model = sm.OLS(y, X).fit()
    rows.append({
        "Asset Class": ac,
        "alpha": model.params.get("const", np.nan),
        "alpha_p": model.pvalues.get("const", np.nan),
        "beta": model.params.get("MktReturn", np.nan),
        "beta_p": model.pvalues.get("MktReturn", np.nan),
        "R2": model.rsquared,
    })

results_table = pd.DataFrame(rows).sort_values("Asset Class").reset_index(drop=True)
results_table

## Deeper Equity Factor Regression

Regress Equity skew portfolio returns on Equity market return plus proxy factor ETFs:
- `MTUM` (momentum)
- `VTV` (value)
- `VUG` (growth)
- `VIG` (quality/dividend proxy)

In [ ]:
factor_tickers = ["MTUM", "VTV", "VUG", "VIG"]
end_date = pd.Timestamp(dt.datetime.now().date()) + pd.Timedelta(days=1)
start_date = end_date - pd.Timedelta(days=HISTORY_DAYS)

factor_frames = []
for t in factor_tickers:
    f = load_ticker_yf(t, start_date, end_date)
    if not f.empty:
        factor_frames.append(f)

if not factor_frames:
    raise ValueError("Could not load factor ETFs from yfinance.")

equity_factors = pd.concat(factor_frames, ignore_index=True)
factor_wide = equity_factors.pivot(index="Date", columns="Ticker", values="LogReturn").reset_index()

equity = asset_portfolios[asset_portfolios["AssetClass"] == "Equity"].copy()
equity = equity.merge(factor_wide, on="Date", how="left")

reg_cols = ["MktReturn", "MTUM", "VTV", "VUG", "VIG"]
reg_df = equity.dropna(subset=["PortfolioReturn"] + reg_cols).copy()

X = sm.add_constant(reg_df[reg_cols])
y = reg_df["PortfolioReturn"]
model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
# Compact coefficient table
coef_table = pd.DataFrame({
    "Coef": model.params,
    "StdErr": model.bse,
    "t": model.tvalues,
    "p": model.pvalues,
    "CI_L": model.conf_int()[0],
    "CI_U": model.conf_int()[1],
}).round(6)
coef_table

In [ ]:
# Basic sanity checks
check = monthly_vals.groupby(["Date", "AssetClass"]).agg(
    long_sum=("SkewWeight", lambda s: s[s > 0].sum()),
    short_sum=("SkewWeight", lambda s: s[s < 0].sum()),
    net=("SkewWeight", "sum"),
).reset_index()

check.describe()